# titulo

introdução

Vamos importar as bibliotecas necewssarias para rodar oque precisamos no notebook

In [1]:
from pathlib import Path
import pandas as pd
from pyreaddbc import dbc2dbf
from dbfread import DBF

Vamos tambem ajustar as opções de exibição para facilitar a analise da base ja que o conjunto de dados possui muitas colunas, e algumas vezes necessitamos de visualizar muitas variaveis disponiveis e evitar que textos longos sejam cortados de forma excessiva

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

Vamos definir os caminhos das pastas utilizadas no processo de conversão.

a pasta "dados/raw" vai conter os arquivos brutos originais em formato dbc, enquanto a pasta "dados/interim", irá armazenar os arquivos intermediarios gerado apos a conversão.

caso a pasta não esxista ela será criada.

In [3]:
pasta_raw = Path("dados/raw")
pasta_interim = Path("dados/interim")

pasta_interim.mkdir(parents=True, exist_ok=True)

Com isso feito, os arquivos dbc, presentes na pasta "dados/raw" serão convertidos para o formato dbf, lidos como dataframes, e então cada arquivo será salvo individualmente em formato csv na pasta "dados/interim", facilitando assim a manipulação posterior tendo em vista que o formato csv é mais simples de se carregar e analisar

In [4]:
for arquivo_dbc in pasta_raw.glob("*.dbc"):
    print(f"Convertendo {arquivo_dbc.name}...")

    arquivo_dbf = pasta_interim / f"{arquivo_dbc.stem}.dbf"
    arquivo_csv = pasta_interim / f"{arquivo_dbc.stem}.csv"

    dbc2dbf(str(arquivo_dbc), str(arquivo_dbf))

    tabela = DBF(
        str(arquivo_dbf),
        encoding="latin1",
        char_decode_errors="ignore"
    )

    df_temp = pd.DataFrame(iter(tabela))

    df_temp.to_csv(
        arquivo_csv,
        index=False,
        sep=";",
        encoding="utf-8-sig"
    )

    print(f"Salvo: {arquivo_csv.name}")

arquivos_csv = list(Path("dados/interim").glob("RD*.csv"))


Convertendo RDMT2001.dbc...
Salvo: RDMT2001.csv
Convertendo RDMT2002.dbc...
Salvo: RDMT2002.csv
Convertendo RDMT2003.dbc...
Salvo: RDMT2003.csv
Convertendo RDMT2004.dbc...
Salvo: RDMT2004.csv
Convertendo RDMT2005.dbc...
Salvo: RDMT2005.csv
Convertendo RDMT2006.dbc...
Salvo: RDMT2006.csv
Convertendo RDMT2007.dbc...
Salvo: RDMT2007.csv
Convertendo RDMT2008.dbc...
Salvo: RDMT2008.csv
Convertendo RDMT2009.dbc...
Salvo: RDMT2009.csv
Convertendo RDMT2010.dbc...
Salvo: RDMT2010.csv
Convertendo RDMT2011.dbc...
Salvo: RDMT2011.csv
Convertendo RDMT2012.dbc...
Salvo: RDMT2012.csv
Convertendo RDMT2101.dbc...
Salvo: RDMT2101.csv
Convertendo RDMT2102.dbc...
Salvo: RDMT2102.csv
Convertendo RDMT2103.dbc...
Salvo: RDMT2103.csv
Convertendo RDMT2104.dbc...
Salvo: RDMT2104.csv
Convertendo RDMT2105.dbc...
Salvo: RDMT2105.csv
Convertendo RDMT2106.dbc...
Salvo: RDMT2106.csv
Convertendo RDMT2107.dbc...
Salvo: RDMT2107.csv
Convertendo RDMT2108.dbc...
Salvo: RDMT2108.csv
Convertendo RDMT2109.dbc...
Salvo: RDMT2

Sendo feita a conversão, os arquivos csv serão carregados e unidos em um unico dataframe, sendo criada tambem a coluna que indica a origem do arquivo, nos permitindo assim identificar qual registro veio de qual arquivo sendo util para conferencia, rastreabilidade e validação desses dados.

In [5]:
dfs = []

for arquivo in arquivos_csv:
    df_temp = pd.read_csv(arquivo, sep=";", encoding="utf-8-sig")
    df_temp["arquivo_origem"] = arquivo.name
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

df.shape

pasta_processed = Path("dados/processed")
pasta_processed.mkdir(parents=True, exist_ok=True)

C:\Users\djmet\AppData\Local\Temp\ipykernel_7452\549544787.py:4: DtypeWarning: Columns (0: DIAGSEC2, 1: DIAGSEC3, 2: DIAGSEC4, 3: DIAGSEC5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(arquivo, sep=";", encoding="utf-8-sig")
C:\Users\djmet\AppData\Local\Temp\ipykernel_7452\549544787.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_temp["arquivo_origem"] = arquivo.name
C:\Users\djmet\AppData\Local\Temp\ipykernel_7452\549544787.py:4: DtypeWarning: Columns (0: DIAGSEC2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(arquivo, sep=";", encoding="utf-8-sig")
C:\Users\djmet\AppData\Local\Temp\ipykernel_7452\549544787.py:5: PerformanceWarning: DataFrame is 

com isso feito, iremos salvar a base consolidada. Essa será a base principal, utilizaa nas proximas etapas do projeto

In [6]:
caminho_base = Path("dados/processed/sih_sus_mt_internacoes_completo.csv")

df = pd.read_csv(
    caminho_base,
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

print(f"Arquivo salvo em: {caminho_base}")

Arquivo salvo em: dados\processed\sih_sus_mt_internacoes_completo.csv


Vamos agora carregar a base consolidade salva a partir do arquivo salvo anteriormente na pasta. a leitura irá utilizar o separador ; e a codificação utf-8-sig, mantendo a compatibilidade com o arquivo exportado.

In [7]:
df = pd.read_csv(
    "dados\processed\sih_sus_mt_internacoes_completo.csv",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

# Vamos exibir a lista de colunas disponiveis para confirmar a extrutura da base
list(df.columns) 

<>:2: SyntaxWarning: invalid escape sequence '\p'
<>:2: SyntaxWarning: invalid escape sequence '\p'
C:\Users\djmet\AppData\Local\Temp\ipykernel_7452\2274088192.py:2: SyntaxWarning: invalid escape sequence '\p'
  "dados\processed\sih_sus_mt_internacoes_completo.csv",


['UF_ZI',
 'ANO_CMPT',
 'MES_CMPT',
 'ESPEC',
 'CGC_HOSP',
 'N_AIH',
 'IDENT',
 'CEP',
 'MUNIC_RES',
 'NASC',
 'SEXO',
 'UTI_MES_IN',
 'UTI_MES_AN',
 'UTI_MES_AL',
 'UTI_MES_TO',
 'MARCA_UTI',
 'UTI_INT_IN',
 'UTI_INT_AN',
 'UTI_INT_AL',
 'UTI_INT_TO',
 'DIAR_ACOM',
 'QT_DIARIAS',
 'PROC_SOLIC',
 'PROC_REA',
 'VAL_SH',
 'VAL_SP',
 'VAL_SADT',
 'VAL_RN',
 'VAL_ACOMP',
 'VAL_ORTP',
 'VAL_SANGUE',
 'VAL_SADTSR',
 'VAL_TRANSP',
 'VAL_OBSANG',
 'VAL_PED1AC',
 'VAL_TOT',
 'VAL_UTI',
 'US_TOT',
 'DT_INTER',
 'DT_SAIDA',
 'DIAG_PRINC',
 'DIAG_SECUN',
 'COBRANCA',
 'NATUREZA',
 'NAT_JUR',
 'GESTAO',
 'RUBRICA',
 'IND_VDRL',
 'MUNIC_MOV',
 'COD_IDADE',
 'IDADE',
 'DIAS_PERM',
 'MORTE',
 'NACIONAL',
 'NUM_PROC',
 'CAR_INT',
 'TOT_PT_SP',
 'CPF_AUT',
 'HOMONIMO',
 'NUM_FILHOS',
 'INSTRU',
 'CID_NOTIF',
 'CONTRACEP1',
 'CONTRACEP2',
 'GESTRISCO',
 'INSC_PN',
 'SEQ_AIH5',
 'CBOR',
 'CNAER',
 'VINCPREV',
 'GESTOR_COD',
 'GESTOR_TP',
 'GESTOR_CPF',
 'GESTOR_DT',
 'CNES',
 'CNPJ_MANT',
 'INFEHOSP',
 'C

resultado por extenso

['UF_ZI',
 'ANO_CMPT',
 'MES_CMPT',
 'ESPEC',
 'CGC_HOSP',
 'N_AIH',
 'IDENT',
 'CEP',
 'MUNIC_RES',
 'NASC',
 'SEXO',
 'UTI_MES_IN',
 'UTI_MES_AN',
 'UTI_MES_AL',
 'UTI_MES_TO',
 'MARCA_UTI',
 'UTI_INT_IN',
 'UTI_INT_AN',
 'UTI_INT_AL',
 'UTI_INT_TO',
 'DIAR_ACOM',
 'QT_DIARIAS',
 'PROC_SOLIC',
 'PROC_REA',
 'VAL_SH',
 'VAL_SP',
 'VAL_SADT',
 'VAL_RN',
 'VAL_ACOMP',
 'VAL_ORTP',
 'VAL_SANGUE',
 'VAL_SADTSR',
 'VAL_TRANSP',
 'VAL_OBSANG',
 'VAL_PED1AC',
 'VAL_TOT',
 'VAL_UTI',
 'US_TOT',
 'DT_INTER',
 'DT_SAIDA',
 'DIAG_PRINC',
 'DIAG_SECUN',
 'COBRANCA',
 'NATUREZA',
 'NAT_JUR',
 'GESTAO',
 'RUBRICA',
 'IND_VDRL',
 'MUNIC_MOV',
 'COD_IDADE',
 'IDADE',
 'DIAS_PERM',
 'MORTE',
 'NACIONAL',
 'NUM_PROC',
 'CAR_INT',
 'TOT_PT_SP',
 'CPF_AUT',
 'HOMONIMO',
 'NUM_FILHOS',
 'INSTRU',
 'CID_NOTIF',
 'CONTRACEP1',
 'CONTRACEP2',
 'GESTRISCO',
 'INSC_PN',
 'SEQ_AIH5',
 'CBOR',
 'CNAER',
 'VINCPREV',
 'GESTOR_COD',
 'GESTOR_TP',
 'GESTOR_CPF',
 'GESTOR_DT',
 'CNES',
 'CNPJ_MANT',
 'INFEHOSP',
 'CID_ASSO',
 'CID_MORTE',
 'COMPLEX',
 'FINANC',
 'FAEC_TP',
 'REGCT',
 'RACA_COR',
 'ETNIA',
 'SEQUENCIA',
 'REMESSA',
 'AUD_JUST',
 'SIS_JUST',
 'VAL_SH_FED',
 'VAL_SP_FED',
 'VAL_SH_GES',
 'VAL_SP_GES',
 'VAL_UCI',
 'MARCA_UCI',
 'DIAGSEC1',
 'DIAGSEC2',
 'DIAGSEC3',
 'DIAGSEC4',
 'DIAGSEC5',
 'DIAGSEC6',
 'DIAGSEC7',
 'DIAGSEC8',
 'DIAGSEC9',
 'TPDISEC1',
 'TPDISEC2',
 'TPDISEC3',
 'TPDISEC4',
 'TPDISEC5',
 'TPDISEC6',
 'TPDISEC7',
 'TPDISEC8',
 'TPDISEC9',
 'arquivo_origem']

Vamos olhar cada coluna mais atentamente e oque cada uma significa

UF_ZI: Município gestor da AIH.

ANO_CMPT: Ano de processamento da AIH.

MES_CMPT: Mês de processamento da AIH.

ESPEC: Especialidade do leito.

CGC_HOSP: CNPJ do estabelecimento hospitalar.

N_AIH: Número da Autorização de Internação Hospitalar.

IDENT: Tipo/identificação da AIH. Ex.: normal, continuação, registro civil, longa permanência.

CEP: CEP do paciente.

MUNIC_RES: Município de residência do paciente.

NASC: Data de nascimento do paciente.

SEXO: Sexo do paciente.

UTI_MES_IN: Dias de UTI no mês em que se iniciou a internação em UTI. Campo antigo/geralmente zerado.

UTI_MES_AN: Dias de UTI no mês anterior ao da alta. Campo antigo/geralmente zerado.

UTI_MES_AL: Dias de UTI no mês da alta. Campo antigo/geralmente zerado.

UTI_MES_TO: Total de dias de UTI durante a internação.

MARCA_UTI: Tipo de UTI utilizada pelo paciente.

UTI_INT_IN: Dias de UTI intermediária no mês de início. Campo antigo/geralmente zerado.

UTI_INT_AN: Dias de UTI intermediária no mês anterior à alta. Campo antigo/geralmente zerado.

UTI_INT_AL: Dias de UTI intermediária no mês da alta. Campo antigo/geralmente zerado.

UTI_INT_TO: Quantidade de diárias em UTI intermediária.

DIAR_ACOM: Quantidade de diárias de acompanhante.

QT_DIARIAS: Quantidade de diárias do paciente.

PROC_SOLIC: Código do procedimento solicitado.

PROC_REA: Código do procedimento realizado.

VAL_SH: Valor de serviços hospitalares.

VAL_SP: Valor de serviços profissionais/prestados por terceiros.

VAL_SADT: Valor de SADT. Campo geralmente zerado em versões recentes.

VAL_RN: Valor de recém-nato. Campo geralmente zerado em versões recentes.

VAL_ACOMP: Valor das diárias de acompanhante. Campo geralmente zerado em versões recentes.

VAL_ORTP: Valor de órtese e prótese. No dicionário aparece como `VAL_ORTOP`; no seu arquivo veio abreviado como `VAL_ORTP`.

VAL_SANGUE: Valor de sangue. Campo geralmente zerado em versões recentes.

VAL_SADTSR: Valor de tomografia/ressonância pagos diretamente a terceiros.

VAL_TRANSP: Valor referente a transplantes/retirada de órgãos.

VAL_OBSANG: Valor de analgesia obstétrica.

VAL_PED1AC: Valor de pediatria para primeira consulta.

VAL_TOT: Valor total da AIH.

VAL_UTI: Valor referente aos gastos em UTI.

US_TOT: Valor total da AIH convertido para dólar.

DT_INTER: Data de internação.

DT_SAIDA: Data de saída/alta.

DIAG_PRINC: Diagnóstico principal segundo CID-10.

DIAG_SECUN: Diagnóstico secundário segundo CID-10. Em versões recentes pode vir zerado.

COBRANCA: Motivo de encerramento/cobrança da AIH.

NATUREZA: Natureza jurídica do hospital. Campo antigo; pode estar zerado em versões recentes.

NAT_JUR: Natureza jurídica do estabelecimento conforme CONCLA.

GESTAO: Tipo de gestão do hospital.

RUBRICA: Rubrica referente à AIH. Campo geralmente não utilizado.

IND_VDRL: Indica execução do exame VDRL.

MUNIC_MOV: Município onde se localiza o hospital.

COD_IDADE: Unidade de medida da idade. Ex.: dias, meses, anos, mais de 100 anos.

IDADE: Idade do paciente na unidade definida por `COD_IDADE`.

DIAS_PERM: Dias de permanência da internação.

MORTE: Indica se a saída ocorreu com morte. 0 = não; 1 = sim.

NACIONAL: Código da nacionalidade do paciente.

NUM_PROC: Número do processamento. Campo antigo/geralmente zerado.

CAR_INT: Caráter da internação. Ex.: eletiva, urgência, acidente de trabalho, trânsito etc.

TOT_PT_SP: Número de pontos de serviços profissionais. Campo antigo/geralmente zerado.

CPF_AUT: CPF do auditor que autorizou pagamento da AIH. Campo antigo/geralmente zerado.

HOMONIMO: Indica se há homônimo do paciente em outra AIH.

NUM_FILHOS: Número de filhos do paciente, usado em casos específicos como laqueadura/vasectomia.

INSTRU: Grau de instrução do paciente.

CID_NOTIF: CID de doença de notificação compulsória.

CONTRACEP1: Tipo de contraceptivo utilizado.

CONTRACEP2: Segundo tipo de contraceptivo utilizado.

GESTRISCO: Indica gestante de risco.

INSC_PN: Número de inscrição da gestante no pré-natal.

SEQ_AIH5: Sequencial da AIH tipo 5, usada para longa permanência.

CBOR: Ocupação do paciente segundo CBO.

CNAER: Atividade econômica relacionada ao paciente.

VINCPREV: Vínculo previdenciário em casos de acidentes/doenças relacionadas ao trabalho.

GESTOR_COD: Motivo de autorização da AIH pelo gestor.

GESTOR_TP: Tipo de gestor.

GESTOR_CPF: CPF do gestor.

GESTOR_DT: Data da autorização dada pelo gestor.

CNES: Código CNES do hospital.

CNPJ_MANT: CNPJ da mantenedora do estabelecimento.

INFEHOSP: Status de infecção hospitalar. Campo antigo/geralmente zerado.

CID_ASSO: CID associado/causa associada.

CID_MORTE: CID da causa de morte.

COMPLEX: Complexidade do procedimento/internação.

FINANC: Tipo de financiamento. Ex.: FAEC ou MAC.

FAEC_TP: Subtipo de financiamento FAEC.

REGCT: Regra contratual.

RACA_COR: Raça/cor do paciente.

ETNIA: Etnia do paciente, quando raça/cor for indígena.

SEQUENCIA: Sequencial da AIH na remessa.

REMESSA: Número da remessa.

AUD_JUST: Justificativa do auditor para aceitação da AIH sem CNS.

SIS_JUST: Justificativa do estabelecimento para aceitação da AIH sem CNS.

VAL_SH_FED: Complemento federal de serviços hospitalares, incluído no valor total da AIH.

VAL_SP_FED: Complemento federal de serviços profissionais, incluído no valor total da AIH.

VAL_SH_GES: Complemento do gestor para serviços hospitalares, incluído no valor total da AIH.

VAL_SP_GES: Complemento do gestor para serviços profissionais, incluído no valor total da AIH.

VAL_UCI: Valor de UCI.

MARCA_UCI: Tipo de UCI utilizada pelo paciente.

DIAGSEC1: Diagnóstico secundário 1.

DIAGSEC2: Diagnóstico secundário 2.

DIAGSEC3: Diagnóstico secundário 3.

DIAGSEC4: Diagnóstico secundário 4.

DIAGSEC5: Diagnóstico secundário 5.

DIAGSEC6: Diagnóstico secundário 6.

DIAGSEC7: Diagnóstico secundário 7.

DIAGSEC8: Diagnóstico secundário 8.

DIAGSEC9: Diagnóstico secundário 9.

TPDISEC1: Tipo do diagnóstico secundário 1.

TPDISEC2: Tipo do diagnóstico secundário 2.

TPDISEC3: Tipo do diagnóstico secundário 3.

TPDISEC4: Tipo do diagnóstico secundário 4.

TPDISEC5: Tipo do diagnóstico secundário 5.

TPDISEC6: Tipo do diagnóstico secundário 6.

TPDISEC7: Tipo do diagnóstico secundário 7.

TPDISEC8: Tipo do diagnóstico secundário 8.

TPDISEC9: Tipo do diagnóstico secundário 9.

arquivo_origem: Coluna criada por você para identificar de qual arquivo mensal o registro veio.


Verificamos agora a estrutura da base, quantidade de registros, nomes das colunas, tipos de dados e uso de memoria

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1044630 entries, 0 to 1044629
Columns: 114 entries, UF_ZI to arquivo_origem
dtypes: float64(30), int64(72), str(12)
memory usage: 948.6 MB


Isso nos mostra que temos:

**1.044.630 registros/linhas**

**114 colunas**

tambem nos indica que temos variaveis numericas e textuais

Agora visualizaremos as 10 primeiras linhas da base, conferindo se os dados foram carregados corretamente, observando o formato das variaveis

In [9]:
df.head(10)

,UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,SEXO,UTI_MES_IN,UTI_MES_AN,UTI_MES_AL,UTI_MES_TO,MARCA_UTI,UTI_INT_IN,UTI_INT_AN,UTI_INT_AL,UTI_INT_TO,DIAR_ACOM,QT_DIARIAS,PROC_SOLIC,PROC_REA,VAL_SH,VAL_SP,VAL_SADT,VAL_RN,VAL_ACOMP,VAL_ORTP,VAL_SANGUE,VAL_SADTSR,VAL_TRANSP,VAL_OBSANG,VAL_PED1AC,VAL_TOT,VAL_UTI,US_TOT,DT_INTER,DT_SAIDA,DIAG_PRINC,DIAG_SECUN,COBRANCA,NATUREZA,NAT_JUR,GESTAO,RUBRICA,IND_VDRL,MUNIC_MOV,COD_IDADE,IDADE,DIAS_PERM,MORTE,NACIONAL,NUM_PROC,CAR_INT,TOT_PT_SP,CPF_AUT,HOMONIMO,NUM_FILHOS,INSTRU,CID_NOTIF,CONTRACEP1,CONTRACEP2,GESTRISCO,INSC_PN,SEQ_AIH5,CBOR,CNAER,VINCPREV,GESTOR_COD,GESTOR_TP,GESTOR_CPF,GESTOR_DT,CNES,CNPJ_MANT,INFEHOSP,CID_ASSO,CID_MORTE,COMPLEX,FINANC,FAEC_TP,REGCT,RACA_COR,ETNIA,SEQUENCIA,REMESSA,AUD_JUST,SIS_JUST,VAL_SH_FED,VAL_SP_FED,VAL_SH_GES,VAL_SP_GES,VAL_UCI,MARCA_UCI,DIAGSEC1,DIAGSEC2,DIAGSEC3,DIAGSEC4,DIAGSEC5,DIAGSEC6,DIAGSEC7,DIAGSEC8,DIAGSEC9,TPDISEC1,TPDISEC2,TPDISEC3,TPDISEC4,TPDISEC5,TPDISEC6,TPDISEC7,TPDISEC8,TPDISEC9,arquivo_origem
0,510000,2020,1,1,NaN,5119103187935,1,78295000,510450,19441102,3,0,0,0,0,0,0,0,0,0,4,4,408050632,408050632,1791.95,247.80,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2039.75,0.0,454.28,20200113,20200117,S721,0,15,0,1236,2,0,0,510250,4,75,4,0,10,NaN,2,0,NaN,0,0,0,NaN,0,0,1,0,0,0,0,0,0,0,0,NaN,2534460,3.507415e+12,NaN,0,0,2,6,NaN,0,3,0,1035,HE51000001N202001.DTS,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0,W000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0,RDMT2001.csv
1,510000,2020,1,1,NaN,5119103187946,1,78250000,510675,19580907,1,0,0,0,0,0,0,0,0,0,6,6,407030026,407030026,559.96,248.61,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,808.57,0.0,180.08,20200113,20200119,K800,0,15,0,1236,2,0,0,510250,4,61,6,0,10,NaN,2,0,NaN,0,0,0,NaN,0,0,1,0,0,0,0,0,0,0,0,NaN,2534460,3.507415e+12,NaN,0,0,2,6,NaN,0,1,0,1036,HE51000001N202001.DTS,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,RDMT2001.csv
2,510000,2020,1,1,NaN,5120100750358,1,78587000,510279,19861212,3,0,0,0,0,0,0,0,0,0,0,1,408020229,408020229,174.57,95.23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,269.80,0.0,60.08,20200129,20200130,S531,0,12,0,1236,2,0,0,510025,4,33,1,0,10,NaN,2,0,NaN,0,0,0,NaN,0,0,1,0,0,0,0,0,0,0,0,NaN,2471345,3.507415e+12,NaN,0,0,2,6,NaN,0,3,0,3471,HE51000001N202001.DTS,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0,W170,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0,RDMT2001.csv
3,510000,2020,1,2,NaN,5120100661159,1,78390000,510170,19920123,3,0,0,0,0,0,0,0,0,0,0,2,303100044,303100044,85.25,23.99,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,109.24,0.0,24.32,20200106,20200108,O623,0,12,0,1236,2,0,0,510170,4,27,2,0,10,NaN,2,0,NaN,0,0,0,NaN,0,0,1,0,0,0,0,0,0,0,0,NaN,2472457,3.507415e+12,NaN,0,0,2,6,NaN,0,99,0,4349,HE51000001N202001.DTS,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,RDMT2001.csv
4,510000,2020,1,2,NaN,5120100661379,1,78390000,510170,19960701,3,0,0,0,0,0,0,0,0,0,0,2,303100044,303100044,85.25,23.99,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,109.24,0.0,24.32,20200112,20200114,O149,0,12,0,1236,2,0,0,510170,4,23,2,0,10,NaN,2,0,NaN,0,0,0,NaN,0,0,1,0,0,0,0,0,0,0,0,NaN,2472457,3.507415e+12,NaN,0,0,2,6,NaN,0,99,0,4350,HE51000001N202001.DTS,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,RDMT2001.csv
5,510000,2020,1,1,NaN,5120100750567,1,78565000,510615,19600117,3,0,0,0,0,0,0,0,0,0,0,1,407030026,407030026,544.36,248.61,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,792.97,0.0,176.60,20200128,20200129,K802,0,12,0,1236,2,0,0,510025,4,60,1,0,10,NaN,1,0,NaN,0,0,0,NaN,0,0,1,0,0,0,0,0,0,0,0,NaN,2471345,3.507415e+12,NaN,0,0,2,6,NaN,0,3,0,3472,HE51000001N202001.DTS,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,RDMT2001.csv
6,510000,2020,1,2,NaN,5120100661445,1,78390000,510170,19950422,3,0,0,0,0,0,0,0,0,0,0,1,303100044,303100044,85.25,23.99,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,109.24,0.0,24.32,20200111,20200111,O234,0,31,0,1236,2,0,0,510170,4,24,0,0,10,NaN,2,0,NaN,0,0,0,NaN,0,0,1,0,0,0,0,0,0,0,

feito isso vamos tambem verificar a auntidade de valores ausentes em cada coluna da base, organizando em uma tabela com o nome da variavel e o total de registros nulos. com isso visamos identificar colunas com muitos campos faltantes, campos antigos, variaveis pouco preenchidas etc

In [10]:
nulos = df.isna().sum().sort_values(ascending=False)

nulos_df = nulos.reset_index()
nulos_df.columns = ["coluna", "qtd_nulos"]

nulos_df

,coluna,qtd_nulos
0,NUM_PROC,1044630
1,CPF_AUT,1044630
2,DIAGSEC8,1044630
3,DIAGSEC9,1044630
4,DIAGSEC6,1044630
5,DIAGSEC7,1044630
6,GESTOR_DT,1044630
7,INFEHOSP,1044630
8,DIAGSEC5,1044625
9,DIAGSEC4,1044605


Agora, iremos verificar a quantidade de valores unicos em cada coluna, visando entender o comportamento das variaveis, identificando colunas com alta cardinalidade, colunas com pouco valores distintos e sem variação. Isso nos ajuda na seleção das variaveis mais adequadas para o projeto

In [11]:
unico = df.nunique().sort_values(ascending=False)

unico_df = unico.reset_index()
unico_df.columns = ["coluna", "qtd_valores_unicos"]

unico_df

,coluna,qtd_valores_unicos
0,N_AIH,1036412
1,VAL_TOT,181521
2,VAL_SH,172259
3,US_TOT,111635
4,VAL_SP,41977
5,NASC,36318
6,CEP,24458
7,SEQUENCIA,11739
8,INSC_PN,9661
9,DIAG_PRINC,6895


Com isso ja temos uma base melhor para trabalhar. temos que ter em vista 3 criterios: 1. Relevancia para a internação prolongada; 2. Disponibilidade antes ou no inicio da internação, visando evitar variaveis que só são conhecidas no fim; 3. qualidade da coluna, ou seja evitar colunas 100% nulas, constantes, identificadores únicos e com risco de vazamento.

com isso em mente, não pretendo usar as variaveis:

N_AIH: é praticamente um identificador unico, e não nos ajudara a generalizar

CEP: ja temos variavel como MUNIC_RES que nos é mais util paradeterminar o local

NASC: idade já tratada é mais util para o modelo

DT_SAIDA: é conhecida no fim da internação, pode vazar no nosso modelo

DIAS_PERM: vou usar para a variavel alvo, se eu deixar vai ocorrer vazamento

MORTE: com todo respeito aos falecidos. como desfecho final da internação se for deixado pode causar vazamento

CID_MORTE:  só fará sentido pos obito e como ja tiramos morte, não tem sentido deixar pelo mesmo motivo

COBRANCA: Motivo de encerramento da AIH. proximo demais do desfecho vindo poder causar vazamento

VAL_TOT, VAL_SH, VAL_SP, VAL_UTI, VAL_UCI, US_TOT: valores financeiros que tendem a refletir duração e gravidade apos a internação

QT_DIARIAS: muito proxima de DIAS_PERM, pode causar vazamento

UTI_MES_TO, UTI_INT_TO, VAL_UTI: podem refletir eventos ocorridos durante a internação não conhecidos no inicio

NUM_PROC, CPF_AUT, DIAGSEC6, DIAGSEC7, DIAGSEC8, DIAGSEC9, GESTOR_DT, INFEHOSP: Colunas totalmente nulas ou sem valores úteis na sua base.

RACA_COR, ETNIA, SEXO: teoricamente poderia ter relevancia pois socialmente grupos diferentes tem estatisticamente desequilibrio socioeconomico, porem isso pode vir a enviesar o modelo barrando em uma barreira etica. já SEXO apesar de poder apresentar relação estatistica com determinados tipos de internação sendo representada por variaveis assistenciais como especialidade do leito, diagnostico principal e procedimento solicitado. alem disso ele pode introduzir simplificações inadequadas, por exemplo em internações obstetricas, ginecologicas ou urologicas, que ja sao identificadas por meio de diagnostico, procedimento e especialidade, sexo faria com que o modelo aprenda associações rigidas ao inves de evitar estimar o risco com base em caracteristicas identitarias.

com isso em mente, vou criar um dataframe com as seguintes variaveis:

DT_INTER: pois vai ajudar a capturar mudanças temporais, porem dividirei em uma variavel para o ano e outra para o mes para facilitar o processamento

IDADE, COD_IDADE: idade do paciente influencia o risco, complexidade e permanencia e COD_IDADE irá ajudar a converter a idade corretamente

MUNIC_RES: Indica municipio onde ocorreu a internação e isso pode ajudar o modelo a enxergar essa tendencia

CNES: indentifica o hospital e os mesmos tem perfis de assistencias diferentes

ESPEC: especialidade do leito, pode influenciar o tempo esperado de internação

CAR_INT: Carater da internação, como urgencia ou eletiva, influenciando a permanencia

DIAG_PRINC: diagnostico principal, sendo uma variavel clinica importante

PROC_SOLIC: procedimento solicitado representa a intenção inicial do tratamento. talvez agrupar possa tratar a cardinalidade

COMPLEX: Indica complexidade da internação, podendo ajudar a diferenciar casos simples e complexos

In [12]:
colunas_projeto = [
    "DT_INTER",
    "IDADE",
    "COD_IDADE",
    "MUNIC_RES",
    "MUNIC_MOV",
    "CNES",
    "ESPEC",
    "CAR_INT",
    "DIAG_PRINC",
    "PROC_SOLIC",
    "COMPLEX",
    "DIAS_PERM"
]

df_projeto = df[colunas_projeto].copy()

df_projeto.head()

,DT_INTER,IDADE,COD_IDADE,MUNIC_RES,MUNIC_MOV,CNES,ESPEC,CAR_INT,DIAG_PRINC,PROC_SOLIC,COMPLEX,DIAS_PERM
0,20200113,75,4,510450,510250,2534460,1,2,S721,408050632,2,4
1,20200113,61,4,510675,510250,2534460,1,2,K800,407030026,2,6
2,20200129,33,4,510279,510025,2471345,1,2,S531,408020229,2,1
3,20200106,27,4,510170,510170,2472457,2,2,O623,303100044,2,2
4,20200112,23,4,510170,510170,2472457,2,2,O149,303100044,2,2


Agora vamos verificar os tipos dos dados e fazer transformações necessarias

In [13]:
print(f"O dataframe tem {df_projeto.shape[1]} colunas e {df_projeto.shape[0]} linhas\n")

print(df_projeto.dtypes)

O dataframe tem 12 colunas e 1044630 linhas

DT_INTER      int64
IDADE         int64
COD_IDADE     int64
MUNIC_RES     int64
MUNIC_MOV     int64
CNES          int64
ESPEC         int64
CAR_INT       int64
DIAG_PRINC      str
PROC_SOLIC    int64
COMPLEX       int64
DIAS_PERM     int64
dtype: object


algumas variaveis  representam codigos administrativos ou clinicos e por isso precisam sertratadas como variaveis categoricas

In [ ]:
colunas_categoricas = [
    "COD_IDADE",
    "MUNIC_RES",
    "MUNIC_MOV",
    "CNES",
    "ESPEC",
    "CAR_INT",
    "DIAG_PRINC",
    "PROC_SOLIC",
    "COMPLEX"
]

for col in colunas_categoricas:
    df_projeto[col] = (
        df_projeto[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

print(df_projeto.dtypes)
df_projeto.head(10)

DT_INTER      int64
IDADE         int64
COD_IDADE       str
MUNIC_RES       str
MUNIC_MOV       str
CNES            str
ESPEC           str
CAR_INT         str
DIAG_PRINC      str
PROC_SOLIC      str
COMPLEX         str
DIAS_PERM     int64
dtype: object


,DT_INTER,IDADE,COD_IDADE,MUNIC_RES,MUNIC_MOV,CNES,ESPEC,CAR_INT,DIAG_PRINC,PROC_SOLIC,COMPLEX,DIAS_PERM
0,20200113,75,4,510450,510250,2534460,1,2,S721,408050632,2,4
1,20200113,61,4,510675,510250,2534460,1,2,K800,407030026,2,6
2,20200129,33,4,510279,510025,2471345,1,2,S531,408020229,2,1
3,20200106,27,4,510170,510170,2472457,2,2,O623,303100044,2,2
4,20200112,23,4,510170,510170,2472457,2,2,O149,303100044,2,2
5,20200128,60,4,510615,510025,2471345,1,1,K802,407030026,2,1
6,20200111,24,4,510170,510170,2472457,2,2,O234,303100044,2,0
7,20200111,17,4,510170,510170,2472457,2,2,O623,303100044,2,0
8,20200130,25,4,510025,510025,2471345,1,2,S065,403010306,2,1
9,20200118,26,4,510025,510025,2471345,2,2,O829,411010034,2,1


podemos observar que os tipos estão coerentes, talvez seja uma ideia converter DT_INTER para um formato datetime, como ela aparemta estar em um formato "%Y%m%d", vou usar ela assim

In [15]:
df_projeto["DT_INTER"] = pd.to_datetime(
    df_projeto["DT_INTER"],
    format="%Y%m%d",
    errors="coerce"
)

print(df_projeto.dtypes)
df_projeto.head(10)

DT_INTER      datetime64[us]
IDADE                  int64
COD_IDADE                str
MUNIC_RES                str
MUNIC_MOV                str
CNES                     str
ESPEC                    str
CAR_INT                  str
DIAG_PRINC               str
PROC_SOLIC               str
COMPLEX                  str
DIAS_PERM              int64
dtype: object


,DT_INTER,IDADE,COD_IDADE,MUNIC_RES,MUNIC_MOV,CNES,ESPEC,CAR_INT,DIAG_PRINC,PROC_SOLIC,COMPLEX,DIAS_PERM
0,2020-01-13,75,4,510450,510250,2534460,1,2,S721,408050632,2,4
1,2020-01-13,61,4,510675,510250,2534460,1,2,K800,407030026,2,6
2,2020-01-29,33,4,510279,510025,2471345,1,2,S531,408020229,2,1
3,2020-01-06,27,4,510170,510170,2472457,2,2,O623,303100044,2,2
4,2020-01-12,23,4,510170,510170,2472457,2,2,O149,303100044,2,2
5,2020-01-28,60,4,510615,510025,2471345,1,1,K802,407030026,2,1
6,2020-01-11,24,4,510170,510170,2472457,2,2,O234,303100044,2,0
7,2020-01-11,17,4,510170,510170,2472457,2,2,O623,303100044,2,0
8,2020-01-30,25,4,510025,510025,2471345,1,2,S065,403010306,2,1
9,2020-01-18,26,4,510025,510025,2471345,2,2,O829,411010034,2,1


Agora vamos verificar dados faltantes ou nulos. caso existam, devemos excluir ou subtituir os valores

In [16]:
print(f'Temos valores nulos? {df_projeto.isnull().values.any()}')

Temos valores nulos? False


teoricamente não temos valores nulos. porem as vezes pode acontecer em variaveis do tipo string, ter coisa escrito que na verdade seria nulo porem como tem algo escrito ele não classifica como nulo, então vamos checar os valores unicos na variavel categorica DIAG_PRINC.

In [17]:
print(f'DIAG_PRINC: {df_projeto['DIAG_PRINC'].unique()}')

DIAG_PRINC: <ArrowStringArray>
['S721', 'K800', 'S531', 'O623', 'O149', 'K802', 'O234', 'S065', 'O829',
 'O800',
 ...
 'D581',  'E63',  'O35', 'A220',  'B39',  'B60', 'E340',  'I87', 'Z139',
 'E242']
Length: 6895, dtype: str


como são muitos valores unicos, vou fazer um calculo de frequencia e gerar um csv para ler melhor em outra aba

In [18]:
freq_diag = (
    df_projeto["DIAG_PRINC"]
    .value_counts(dropna=False)
    .reset_index()
)

freq_diag.columns = ["DIAG_PRINC", "quantidade"]

freq_diag.to_csv(
    "dados/processed/frequencia_diag_princ.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

Apos analisar o csv "frequencia_diag_princ", não foi identificado nenhum valor invalido ou nulo ou seja, não temos valores nulos e nem com erro de digitação

Tendo feito isso, vamos criar algumas variaveis que facilitarão o processamento do modelo

vamos usar DT_INTER para criar duas novas variaveis: ano_internacao e mes_internacao, que nos permitirão representar o momento da internação de forma mais simples e nos permitindo analisar melhor os padroes temporais como sazonalidade por exemplo

In [19]:
df_projeto["ano_internacao"] = df_projeto["DT_INTER"].dt.year
df_projeto["mes_internacao"] = df_projeto["DT_INTER"].dt.month

Vamos usar DIAG_PRINC para criar a variavel cid3, que representa apenas os 3 primeiros digitos do cid reduzindo assim a quantidade de categorias do diagnostico peincipal, e as colocando no nivel inicial do CID-10, diminuindo assim sua complexidade e ajudando a capturar padrões clinicos mais gerais

In [20]:
df_projeto["cid3"] = df_projeto["DIAG_PRINC"].str[:3]

Vamos usar PROC_SOLIC para criar a variavel grupo_procedimento, que representa os dois primeiros caracteres de PROC_SOLIC. com isso agrupamos procedimentos semelhantes em categorias mais amplas, utilizando a informação do procedimento solicitado sem lidar com uma quantidade escessiva de codigos individuais

In [21]:
df_projeto["grupo_procedimento"] = df_projeto["PROC_SOLIC"].str[:2]

Vamos usar as variaveis IDADE e COD_IDADE para criar a variavel idade_anos, convertendo assim a idade do paciente para uma unidade padronizada em anos. a base utiliza a variavel COD_IDADE para indicar se a idade esta registrada em dias, meses, anos ou 100 anos ou mais, por isso usamos essa função para interpretar corretamente essa unidade e transformar a idade em um formato mais adequado para analise e modelagem

In [22]:
def converter_idade_anos(row):
    cod = str(row["COD_IDADE"])
    idade = row["IDADE"]

    if pd.isna(idade):
        return None

    if cod == "2":      # idade em dias
        return idade / 365
    elif cod == "3":    # idade em meses
        return idade / 12
    elif cod == "4":    # idade em anos
        return idade
    elif cod == "5":    # 100 anos ou mais
        return 100
    else:
        return None

df_projeto["idade_anos"] = df_projeto.apply(converter_idade_anos, axis=1)

Agora criaremos a variavel alvo a partir de DIAS_PERM. Como o objetivo é prever internações prolongadas, vamos utilizar o terceiro quartil dda variavel DIAS_PERM para que as internações acima desse limite sejam classificadas como "1", ou seja prolongadas, e as demais como "0". isso nos permite transformar o ploblema em uma tarefa de classificação binaria facilitando a aplicação de modelos de machine learning, limitando assim tambem a escolha arbitraria de um numero de dias pelo modelo pois o limite será definido com base na propia distribuição dos dados. com isso foi construida a variavel binaria internacao_prolongada

In [23]:
limite_permanencia = df_projeto["DIAS_PERM"].quantile(0.75)

df_projeto["internacao_prolongada"] = (
    df_projeto["DIAS_PERM"] > limite_permanencia
).astype(int)

Agora observamos para ver a tabela

In [24]:
df_projeto.head(10)

,DT_INTER,IDADE,COD_IDADE,MUNIC_RES,MUNIC_MOV,CNES,ESPEC,CAR_INT,DIAG_PRINC,PROC_SOLIC,COMPLEX,DIAS_PERM,ano_internacao,mes_internacao,cid3,grupo_procedimento,idade_anos,internacao_prolongada
0,2020-01-13,75,4,510450,510250,2534460,1,2,S721,408050632,2,4,2020,1,S72,40,75.0,0
1,2020-01-13,61,4,510675,510250,2534460,1,2,K800,407030026,2,6,2020,1,K80,40,61.0,1
2,2020-01-29,33,4,510279,510025,2471345,1,2,S531,408020229,2,1,2020,1,S53,40,33.0,0
3,2020-01-06,27,4,510170,510170,2472457,2,2,O623,303100044,2,2,2020,1,O62,30,27.0,0
4,2020-01-12,23,4,510170,510170,2472457,2,2,O149,303100044,2,2,2020,1,O14,30,23.0,0
5,2020-01-28,60,4,510615,510025,2471345,1,1,K802,407030026,2,1,2020,1,K80,40,60.0,0
6,2020-01-11,24,4,510170,510170,2472457,2,2,O234,303100044,2,0,2020,1,O23,30,24.0,0
7,2020-01-11,17,4,510170,510170,2472457,2,2,O623,303100044,2,0,2020,1,O62,30,17.0,0
8,2020-01-30,25,4,510025,510025,2471345,1,2,S065,403010306,2,1,2020,1,S06,40,25.0,0
9,2020-01-18,26,4,510025,510025,2471345,2,2,O829,411010034,2,1,2020,1,O82,41,26.0,0


Agora criamos uma nova base apenas com as colunas que iremos usar

In [25]:
variaveis_modelo = [
    "ano_internacao",
    "mes_internacao",
    "idade_anos",
    "MUNIC_RES",
    "MUNIC_MOV",
    "CNES",
    "ESPEC",
    "CAR_INT",
    "cid3",
    "grupo_procedimento",
    "COMPLEX",
    "internacao_prolongada"
]

df_modelo = df_projeto[variaveis_modelo].copy()

df_modelo.head(10)

,ano_internacao,mes_internacao,idade_anos,MUNIC_RES,MUNIC_MOV,CNES,ESPEC,CAR_INT,cid3,grupo_procedimento,COMPLEX,internacao_prolongada
0,2020,1,75.0,510450,510250,2534460,1,2,S72,40,2,0
1,2020,1,61.0,510675,510250,2534460,1,2,K80,40,2,1
2,2020,1,33.0,510279,510025,2471345,1,2,S53,40,2,0
3,2020,1,27.0,510170,510170,2472457,2,2,O62,30,2,0
4,2020,1,23.0,510170,510170,2472457,2,2,O14,30,2,0
5,2020,1,60.0,510615,510025,2471345,1,1,K80,40,2,0
6,2020,1,24.0,510170,510170,2472457,2,2,O23,30,2,0
7,2020,1,17.0,510170,510170,2472457,2,2,O62,30,2,0
8,2020,1,25.0,510025,510025,2471345,1,2,S06,40,2,0
9,2020,1,26.0,510025,510025,2471345,2,2,O82,41,2,0
